<a href="https://colab.research.google.com/github/BreakTheAlgorithm/MLforALL/blob/main/book_ch10_ensemble_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🌳 Chapter 10 — Ensemble Methods</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 75 mins | Level: Intermediate–Advanced</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Explain bagging, boosting, and stacking
- Train Random Forest, AdaBoost, Gradient Boosting, and XGBoost
- Interpret feature importance from tree-based ensembles
- Understand how each method reduces bias or variance
- Build a stacking ensemble from scratch

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets        import make_classification
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import (
    RandomForestClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, BaggingClassifier, VotingClassifier
)
from sklearn.linear_model    import LogisticRegression
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import f1_score, accuracy_score

%matplotlib inline
np.random.seed(42)

X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42, class_sep=0.7
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 📖 Section A — Ensemble Strategies

| Method | Idea | Reduces | Example |
|--------|------|---------|--------|
| **Bagging** | Train N models on N bootstrap samples, average | Variance | Random Forest |
| **Boosting** | Train models sequentially, each fixing previous errors | Bias | AdaBoost, Gradient Boosting |
| **Stacking** | Train meta-model on predictions of base models | Both | Custom stacks |

> 💡 **Key Rule:** Bagging works best when base models have HIGH VARIANCE (deep trees). Boosting works best when base models have HIGH BIAS (shallow trees/stumps). Mixing them is the art.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: All Ensemble Methods Compared
# ─────────────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ensembles = {
    'Single Decision Tree':  DecisionTreeClassifier(random_state=42),
    'Bagging (DT)':          BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost':              AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print(f'{"Method":25s}  {"CV F1 Mean":>12}  {"CV F1 Std":>12}')
print('-' * 54)
for name, m in ensembles.items():
    scores = cross_val_score(m, X, y, cv=cv, scoring='f1')
    print(f'{name:25s}  {scores.mean():12.4f}  {scores.std():12.4f}')

---
## 🟢 Exercise 10.1 — Feature Importance *(Level 1 · Est. 10 min)*

> Train a Random Forest. Extract and plot feature importances. Print top 5 features.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 10.1: Feature Importance
# ─────────────────────────────────────────────────────────────
feature_names = [f'feature_{i}' for i in range(X.shape[1])]

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Extract, sort, and plot feature importances
# YOUR CODE HERE

print('Top 5 features:')
# YOUR CODE HERE

assert hasattr(rf, 'feature_importances_')
assert len(rf.feature_importances_) == X.shape[1]
print('\n✅ Feature importance extracted!')

In [ ]:
# @title ✅ Solution — Exercise 10.1 (click to expand)
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 5 features:')
print(importance_df.head())

plt.figure(figsize=(10, 5))
plt.bar(importance_df['feature'].head(10), importance_df['importance'].head(10), color='steelblue')
plt.title('Top 10 Feature Importances — Random Forest')
plt.xticks(rotation=30)
plt.ylabel('Importance')
plt.tight_layout()
plt.show()
print('✅ Feature importance reveals which features contribute most to splits across all trees.')

---
## 🟡 Exercise 10.2 — Effect of n_estimators on RF Performance *(Level 2 · Est. 15 min)*

> Plot test F1 vs n_estimators for Random Forest (range 1–200). Find the 'elbow point' where adding more trees stops helping.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 10.2: n_estimators Sweep
# ─────────────────────────────────────────────────────────────
n_estimator_range = [1, 5, 10, 20, 50, 75, 100, 150, 200]
f1_scores = []

for n in n_estimator_range:
    # YOUR CODE HERE
    pass

plt.figure(figsize=(9, 5))
# YOUR CODE HERE — plot
plt.title('Random Forest: F1 vs n_estimators')
plt.xlabel('Number of Trees')
plt.ylabel('Test F1 Score')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# @title ✅ Solution — Exercise 10.2 (click to expand)
n_estimator_range = [1, 5, 10, 20, 50, 75, 100, 150, 200]
f1_scores = []

for n in n_estimator_range:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_n.fit(X_train, y_train)
    yp = rf_n.predict(X_test)
    f1_scores.append(f1_score(y_test, yp))

plt.figure(figsize=(9, 5))
plt.plot(n_estimator_range, f1_scores, 'bo-', markersize=7)
plt.title('Random Forest: F1 vs n_estimators')
plt.xlabel('Number of Trees')
plt.ylabel('Test F1 Score')
plt.grid(True, alpha=0.3)
plt.show()
print(f'Best n_estimators: {n_estimator_range[np.argmax(f1_scores)]}  (F1={max(f1_scores):.4f})')
print('\n✅ F1 typically stabilises around 50-100 trees. Adding more helps diminishing returns — and costs compute time.')

---
## 🔴 Exercise 10.3 — Build a Stacking Ensemble *(Level 3 · Est. 25 min)*

> Build a manual stacking ensemble: base models (LR, KNN, DT), meta-model (Logistic Regression). Use out-of-fold predictions to generate meta-features. Compare to individual models.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 10.3: Manual Stacking Ensemble
# ─────────────────────────────────────────────────────────────

base_models = [
    ('lr',  LogisticRegression(max_iter=1000)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('dt',  DecisionTreeClassifier(max_depth=5, random_state=42)),
]

# Step 1: Generate out-of-fold meta-features
# YOUR CODE HERE

# Step 2: Train meta-model on meta-features
# YOUR CODE HERE

# Step 3: Predict on test set
# YOUR CODE HERE

print('Stacking ensemble F1:', f1_score(y_test, stacking_preds))
print('\nComparison:')
for name, m in base_models:
    m.fit(X_train_s, y_train)
    print(f'  {name}: F1={f1_score(y_test, m.predict(X_test_s)):.4f}')

In [ ]:
# @title ✅ Solution — Exercise 10.3 (click to expand)

from sklearn.model_selection import StratifiedKFold

base_models = [
    ('lr',  LogisticRegression(max_iter=1000)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('dt',  DecisionTreeClassifier(max_depth=5, random_state=42)),
]
meta_model = LogisticRegression(max_iter=1000)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Step 1: Generate out-of-fold predictions (meta-features)
oof_preds = np.zeros((len(X_train_s), len(base_models)))
for j, (name, m) in enumerate(base_models):
    for train_idx, val_idx in kf.split(X_train_s, y_train):
        m.fit(X_train_s[train_idx], y_train[train_idx])
        oof_preds[val_idx, j] = m.predict_proba(X_train_s[val_idx])[:, 1]

# Step 2: Train meta-model on OOF predictions
meta_model.fit(oof_preds, y_train)

# Step 3: Test predictions — refit base models on full train, predict test
test_meta = np.zeros((len(X_test_s), len(base_models)))
for j, (name, m) in enumerate(base_models):
    m.fit(X_train_s, y_train)
    test_meta[:, j] = m.predict_proba(X_test_s)[:, 1]
stacking_preds = meta_model.predict(test_meta)

print(f'Stacking ensemble F1: {f1_score(y_test, stacking_preds):.4f}')
print('\nBase model comparisons:')
for name, m in base_models:
    print(f'  {name}: F1={f1_score(y_test, m.predict(X_test_s)):.4f}')
print('\n✅ Stacking combines complementary model strengths — the meta-model learns when to trust each base model.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: Why does Random Forest reduce variance compared to a single Decision Tree?</strong></summary>

**Answer:** A single deep decision tree has HIGH VARIANCE — it fits training data very closely (memorises it). Random Forest builds N trees, each on a bootstrap sample of the data AND a random subset of features. The final prediction is the majority vote. Because the trees are built on different data and features, their errors are uncorrelated. When you average N uncorrelated models, variance decreases by ~1/N. This is the mathematical intuition behind bagging.
</details>

<details>
<summary><strong>Q2: What is the difference between AdaBoost and Gradient Boosting?</strong></summary>

**Answer:** AdaBoost reweights training samples — misclassified samples get higher weight so the next model focuses on them. Gradient Boosting instead fits each model to the RESIDUALS (gradient of the loss) of the previous model's prediction. Gradient Boosting is more general (works for any differentiable loss), and typically achieves better performance. XGBoost/LightGBM are highly optimised implementations of Gradient Boosting that dominate Kaggle competitions.
</details>

<details>
<summary><strong>Q3: What is the risk of boosting vs bagging?</strong></summary>

**Answer:** Boosting sequentially corrects errors — but if early models are very wrong, late models can overfit the corrections. Boosting is more prone to overfitting than bagging. Bagging is robust to overfitting because trees are independent. Practical rule: for noisy data, prefer Random Forest (bagging). For clean, structured data with a hard accuracy requirement, use Gradient Boosting with careful regularisation (max_depth, learning_rate, n_estimators).
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 10 Complete!</h3>
<ul>
<li>Bagging (Random Forest), Boosting (AdaBoost, Gradient Boosting), Stacking</li>
<li>Feature importance from tree ensembles</li>
<li>Effect of n_estimators on performance</li>
<li>Manual stacking ensemble with out-of-fold predictions</li>
</ul>
<p><strong>Next:</strong> Chapter 11 — Unsupervised Learning: Clustering</p>
</div>